
# 04 — Mixture-of-Experts (MoE) Routing, from Scratch

**Goal:** implement a sparse MoE feed-forward layer (router + top-k dispatch + a load-balancing
auxiliary loss), verify the routing and loss behave as claimed, and be fluent on why sparse MoE
decouples parameter count from compute, what happens without load balancing, and why MoE is a
serving-memory concern much like the KV cache is. MoE architectures (Mixtral, DeepSeek, Qwen-MoE,
etc.) are now common enough that "how does MoE routing work" is a standard interview question.

Structure: **Lesson → Implementation → Quiz → Final Answers & Explanations.**



## 1. Lesson

### 1.1 The idea: sparse activation

A dense transformer's FFN block runs the *same* feed-forward network on every token. A
Mixture-of-Experts layer instead has $N$ separate "expert" FFNs (usually the same architecture
as a normal FFN, just many independent copies) and a small **router** network that, per token,
decides which *few* experts should process that token. Only the selected experts actually run
for a given token — so total parameter count scales with $N$ (you can have dozens of experts),
while the compute (FLOPs) per token only scales with however many experts are *actually* used per
token (top-k, usually $k=1$ or $k=2$), not with $N$. This is the core appeal: **you can scale
parameter count far ahead of compute cost**, which is why MoE models can have far more total
parameters than a dense model of equivalent inference cost.

### 1.2 The router and top-k gating

For each token's hidden state $x$, the router computes logits over experts:
$$
\text{logits} = x W_{router}, \qquad p = \text{softmax}(\text{logits}) \in \mathbb{R}^N
$$
Then select the top-$k$ experts by probability, renormalize their probabilities to sum to 1
(since you're not using all $N$), and combine each selected expert's output weighted by its
(renormalized) gate probability:
$$
\text{out} = \sum_{i \in \text{top-}k(p)} \tilde{p}_i \cdot \text{Expert}_i(x)
$$
- **Top-1** (Switch Transformer style): route each token to exactly one expert. Simplest, cheapest,
  but a single hard routing decision per token with no ability to blend.
- **Top-2** (Mixtral style, among others): route to two experts and blend — a bit more compute,
  generally better quality, and slightly more forgiving of any single router mistake since a
  second expert's output still contributes.

### 1.3 The load-balancing problem

Nothing in the basic router objective *by itself* prevents the router from collapsing to always
picking the same 1-2 "favorite" experts for every token (a classic pathology: if one expert
happens to get slightly better early in training, it gets more gradient signal, gets even better,
and the router routes to it even more — a rich-get-richer feedback loop). This is bad for two
reasons: (a) most experts end up undertrained/useless, wasting the extra parameters MoE was
supposed to buy you, and (b) it defeats load-balancing across devices in distributed training/
serving, where each expert typically lives on a different accelerator — if all tokens pile onto
one expert, that device becomes a bottleneck while others sit idle.

The fix: an auxiliary **load-balancing loss**, added to the training objective, that penalizes
uneven routing. A standard formulation (Switch Transformer-style):
$$
\mathcal{L}_{balance} = N \sum_{i=1}^{N} f_i \cdot P_i
$$
where $f_i$ is the fraction of tokens actually dispatched to expert $i$, and $P_i$ is the average
router probability mass expert $i$ received across the batch. This product $f_i P_i$ is minimized
(for a fixed total, by AM-GM-style reasoning) when routing is close to uniform across experts —
so gradient descent on this auxiliary loss actively pushes the router toward balanced usage,
counteracting the rich-get-richer dynamic.

### 1.4 Capacity and token dropping

In real (especially distributed) MoE systems, each expert is given a fixed **capacity** — a
maximum number of tokens it will process in a batch (needed because you can't dynamically resize
a device's compute buffer mid-batch). If more tokens route to an expert than its capacity allows,
the excess tokens are **dropped** for that expert (typically passed through via a residual
connection with no expert contribution, or routed to a fallback), directly hurting those specific
tokens' quality. This ties capacity factor choice to a real trade-off: too low and you drop (and
degrade) tokens; too high and you waste allocated compute/memory that goes unused whenever
routing isn't perfectly even.

### 1.5 The systems angle: memory footprint

Even though only a few experts run per *token*, **all** experts' weights must be resident in
memory (or quickly reachable) to serve the model at all, since different tokens in the same batch
(or different requests) will route to different experts. So MoE's parameter-count-vs-compute
decoupling (1.1) is a genuine inference-compute win, but not a memory win — total memory
footprint scales with total parameters across all experts, same as a dense model of that many
total parameters would. This is directly analogous to the KV-cache memory-vs-compute framing:
both are "you don't compute against all of it every step, but you still have to keep all of it
around" situations that shape real serving-system design.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)



## 2. Implementation

Implement a small `MoELayer` with top-k routing, plus a `load_balancing_loss` function. Fill in
every `# TODO`. We use a simple (non-capacity-limited) dispatch for clarity — capacity/dropping
is discussed in the lesson and quiz but not implemented, to keep the core routing logic legible.

- `Expert`: a small 2-layer FFN (`Linear -> ReLU -> Linear`), already implemented for you below.
- `MoELayer.forward(x)`, `x`: `(batch, seq, d_model)`. Returns `(output, probs, topk_idx)` where
  `output` is `(batch, seq, d_model)`, `probs` is the *full* softmax over all `num_experts` (per
  token, before top-k selection — needed for the load-balancing loss), and `topk_idx` is
  `(N, top_k)` with `N = batch*seq`, the indices of the experts each token was routed to.
- `load_balancing_loss(probs, topk_idx, num_experts)`: implements
  $N_{experts}\sum_i f_i P_i$ as described in the lesson.


In [ ]:

class Expert(nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_model))

    def forward(self, x):
        return self.net(x)


class MoELayer(nn.Module):
    def __init__(self, d_model: int, d_hidden: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        # TODO: self.router = nn.Linear(d_model, num_experts)
        self.router = None
        self.experts = nn.ModuleList([Expert(d_model, d_hidden) for _ in range(num_experts)])

    def forward(self, x: torch.Tensor):
        orig_shape = x.shape
        x_flat = x.reshape(-1, orig_shape[-1])  # (N, d_model)
        N = x_flat.shape[0]

        # TODO: logits = self.router(x_flat) -> (N, num_experts)
        # TODO: probs = softmax(logits, dim=-1) -> (N, num_experts)  [the FULL distribution]
        logits = None
        probs = None

        # TODO: topk_probs, topk_idx = probs.topk(self.top_k, dim=-1)  -> each (N, top_k)
        # TODO: renormalize topk_probs so each token's top-k gate weights sum to 1
        topk_probs = None
        topk_idx = None

        out = torch.zeros_like(x_flat)
        # TODO: for each of the top_k slots, and each expert, find which tokens in this slot
        # were routed to that expert (a boolean mask), and accumulate
        # gate_weight * expert(x_flat) into `out` for exactly those tokens.
        # (A double loop over top_k and num_experts with boolean masking is fine here --
        # clarity over cleverness.)

        out = out.reshape(orig_shape)
        return out, probs, topk_idx


def load_balancing_loss(probs: torch.Tensor, topk_idx: torch.Tensor, num_experts: int) -> torch.Tensor:
    \"\"\"
    probs: (N, num_experts) full softmax distribution per token (NOT just top-k)
    topk_idx: (N, top_k) indices of experts actually dispatched to, per token
    Returns a scalar: num_experts * sum_i f_i * P_i
      where f_i = fraction of tokens with expert i in their top-k (dispatch fraction)
            P_i = average router probability mass expert i received across all tokens
    \"\"\"
    N = probs.shape[0]
    # TODO: one_hot = F.one_hot(topk_idx, num_classes=num_experts).float()  -> (N, top_k, num_experts)
    # TODO: dispatch = one_hot summed over the top_k dim, clamped to max 1.0 -> (N, num_experts)
    #       (clamp handles the (impossible here, but general) case of an expert appearing twice)
    # TODO: f = dispatch.mean(dim=0)  -> (num_experts,)
    # TODO: P = probs.mean(dim=0)    -> (num_experts,)
    # TODO: return num_experts * (f * P).sum()
    raise NotImplementedError



### Sanity tests

1. **Shape check.**
2. **Routing sanity**: each token's `topk_idx` row should contain `top_k` *distinct* expert
   indices (softmax top-k never repeats an index for a single token).
3. **The big one — load-balancing loss actually penalizes imbalance**: construct a deliberately
   uniform routing distribution and a deliberately skewed one, and confirm the uniform case gets
   a strictly lower loss. If your implementation just computes *something* without this
   property, it isn't doing its job.


In [ ]:

d_model, d_hidden, num_experts, top_k = 8, 16, 4, 2
moe = MoELayer(d_model, d_hidden, num_experts, top_k)
x = torch.randn(2, 5, d_model)

out, probs, topk_idx = moe(x)
assert out.shape == x.shape
assert probs.shape == (10, num_experts)
assert topk_idx.shape == (10, top_k)
print("Test 1 passed: shapes correct")


In [ ]:

for row in topk_idx:
    assert len(set(row.tolist())) == top_k, f"expected {top_k} distinct experts, got {row.tolist()}"
print("Test 2 passed: every token routed to exactly top_k distinct experts")


In [ ]:

N = 100
uniform_probs = torch.ones(N, num_experts) / num_experts
skewed_probs = torch.zeros(N, num_experts)
skewed_probs[:, 0] = 0.7
skewed_probs[:, 1:] = 0.3 / (num_experts - 1)

uniform_topk = uniform_probs.topk(top_k, dim=-1).indices
skewed_topk = skewed_probs.topk(top_k, dim=-1).indices

loss_uniform = load_balancing_loss(uniform_probs, uniform_topk, num_experts)
loss_skewed = load_balancing_loss(skewed_probs, skewed_topk, num_experts)
print(f"loss (uniform routing) = {loss_uniform.item():.3f}")
print(f"loss (skewed routing)  = {loss_skewed.item():.3f}")
assert loss_uniform.item() < loss_skewed.item(), \
    "load-balancing loss should be LOWER for uniform routing than for skewed routing"
print("Test 3 passed: load-balancing loss correctly penalizes uneven routing")



## 3. Quiz

1. Why does sparse MoE let you scale total parameter count without a proportional increase in
   per-token compute? What's the precise thing that stays "dense" per token and the thing that
   doesn't?
2. What actually goes wrong, mechanistically, if you train an MoE router *without* a
   load-balancing loss? Why is it a feedback loop, not just a one-time imbalance?
3. Top-1 vs. top-2 routing — what do you gain by going to top-2, and what does it cost you?
4. What is "capacity" in an MoE layer, and what happens to tokens that exceed an expert's
   capacity? Why does this create a trade-off in choosing the capacity factor?
5. Why is MoE, despite its compute savings, *not* a memory-footprint win versus a dense model
   with the same active-per-token compute? Draw the parallel to KV-cache memory explicitly.
6. If you were told an MoE model is a good fit for a scenario with abundant memory but limited
   compute/latency budget, why does that pairing make sense given everything above?
7. The router's top-k selection is a discrete, non-differentiable operation (choosing which
   experts to use). How does gradient descent still manage to train the router at all?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — Parameter count vs. compute
Total parameter count in an MoE layer scales with $N$ (the number of experts), because every
expert's full set of weights exists and must be stored, regardless of how often it's used. But
per-token *compute* only involves the top-$k$ experts actually selected for that token — the
router and the (small number of) selected experts' forward passes are the only FLOPs paid per
token, entirely independent of how large $N$ is. So you can grow $N$ (total capacity/parameters)
arbitrarily while the per-token compute cost stays pinned to whatever $k$ (and the per-expert
FFN size) you chose — the thing that stays "dense" per token is the compute through $k$ experts;
the thing that scales freely is the *total* number of experts (and hence total parameters) that
exist but sit idle for any given token.

### Q2 — Why load balancing is a feedback loop, not one-time
Without a balancing term, the only signal the router gets is "which expert produces the lowest
task loss for this token" — and early in training, if one expert happens to be marginally better
(random init luck, or slightly more capacity used early), the router starts sending it more
tokens. More tokens means more gradient updates to that expert, which makes it *even* better
relative to the others, which makes the router prefer it *even more* — a positive feedback loop
(sometimes called "expert collapse") that can converge to routing nearly everything to a small
subset of experts, leaving the rest chronically undertrained and effectively wasted parameters.
It's a feedback loop rather than a one-time imbalance precisely because the router's preference
and the expert's quality reinforce each other every training step, not just at initialization.

### Q3 — Top-1 vs top-2
Top-1 is cheapest (only one expert's FFN runs per token) and simplest to implement/serve, but
commits every token's processing to a single expert's decision — if the router's top pick is
subtly wrong for a token, there's no second opinion blended in. Top-2 roughly doubles the
per-token FFN compute (two experts run and are blended by their renormalized gate weights) but
generally improves quality: blending two experts' outputs is more robust to any single routing
mistake, and gives the model a smoother way to represent tokens that plausibly belong to more
than one "specialty." The choice is a direct compute-vs-quality dial, same shape as many
architecture trade-offs elsewhere in this material.

### Q4 — Capacity and token dropping
"Capacity" is a fixed maximum number of tokens an expert will process within a batch/step — set
in advance because real (especially distributed, multi-device) systems need a fixed-size compute
buffer per expert rather than one that dynamically resizes based on how many tokens happen to
route there. If routing sends *more* tokens to an expert than its capacity allows, the excess
tokens are dropped for that expert (commonly passed through via a residual/skip connection with
no expert contribution at all), directly degrading quality for exactly those tokens. This creates
a trade-off in the **capacity factor** (capacity as a multiple of the "perfectly even" per-expert
load): set it too low and routing imbalance (which, per Q2, never perfectly vanishes even with a
balancing loss) causes frequent token drops and quality loss; set it too high and you allocate
compute/memory buffer space that mostly goes unused, wasting resources for a rare worst case.

### Q5 — Not a memory win
Sparse MoE saves *compute* per token (only $k$ of $N$ experts run), but every expert's weights
still have to be resident in memory (or quickly loadable) to serve the model at all, since which
experts get used varies token-by-token and request-by-request across a batch — you can't predict
in advance which experts you'll need and load only those. So total memory footprint scales with
*all* experts' combined parameter count, exactly like a dense model with that many total
parameters would. This is structurally identical to the KV-cache situation (notebook 07): a KV
cache also doesn't require recomputing the full history every step (a compute saving), but the
full cached history still has to sit in memory the whole time (no compute saving translates into
a memory saving) — both are examples of "you only *touch* a subset per step, but you must *keep*
the whole thing around."

### Q6 — Why MoE fits "abundant memory, limited compute" scenarios
Given Q1 and Q5: MoE gives you a large total parameter budget (which tends to correlate with
model capability/quality) while keeping the *compute* cost per token pinned to a small, chosen
active-expert count — exactly the profile you want when you have plenty of memory/storage to
hold many experts' weights but a tight latency or FLOPs budget per request. A dense model with
the same *quality* would typically need to actually run all of its (larger) parameter count's
worth of compute per token; MoE decouples those two axes, letting you buy quality with memory
rather than with per-token compute, which is precisely the resource trade being made.

### Q7 — Training through a discrete routing decision
The top-k *selection* (which specific experts get used) is indeed a hard, non-differentiable
choice — but gradients still flow to the router in two ways: (a) through the **gate weights**
themselves (the renormalized softmax probabilities multiplying each selected expert's output) —
these are continuous and differentiable, so the router learns to adjust *how much* weight to
give an expert once it's selected, and via the softmax, nearby experts' probabilities also shift
smoothly as router logits change; and (b) through the **auxiliary load-balancing loss**, which is
computed from the full (differentiable) softmax distribution over all experts and directly
provides gradient signal shaping which experts the router tends to prefer overall, independent of
the hard top-k cutoff. In practice, this combination (differentiable gate weights + an auxiliary
loss over the full distribution) is what lets standard backprop train the router adequately
despite the discrete top-k step in the middle — there's no need for reinforcement-learning-style
gradient estimators here, though some MoE variants do explore soft/differentiable routing
relaxations as an active research area.
